In [19]:
import sys
import os
import pandas as pd
from pathlib import Path

In [20]:
# Project setup
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "raw").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [21]:
from common.spark_session import get_spark

spark = get_spark("exploration")
RAW = PROJECT_ROOT / "data" / "raw"
print("Working directory set to:", os.getcwd())

Working directory set to: f:\Sem 3A\Data-intensive Computing\urban-data-platform


In [22]:
# Load / inspect CSV files

for name in ["air_quality", "weather", "taxi_zones"]:
    files = list((RAW / name).glob("*.csv"))
    print(f"\n=== {name}: {[f.name for f in files]} ===")

    df = pd.read_csv(files[0], nrows=1000)

    print("columns:", list(df.columns))
    print(df.head(3).to_string())


=== air_quality: ['hourly_88101_2024.csv'] ===
columns: ['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Date Local', 'Time Local', 'Date GMT', 'Time GMT', 'Sample Measurement', 'Units of Measure', 'MDL', 'Uncertainty', 'Qualifier', 'Method Type', 'Method Code', 'Method Name', 'State Name', 'County Name', 'Date of Last Change']
   State Code  County Code  Site Num  Parameter Code  POC   Latitude  Longitude  Datum            Parameter Name  Date Local Time Local    Date GMT Time GMT  Sample Measurement             Units of Measure  MDL  Uncertainty  Qualifier Method Type  Method Code                                                            Method Name State Name County Name Date of Last Change
0           1            3        10           88101    3  30.497478 -87.880258  NAD83  PM2.5 - Local Conditions  2024-01-02      14:00  2024-01-02    20:00                 9.0  Micrograms/cubic meter (LC)    5          NaN 

In [23]:

# Taxi Parquet

taxi_files = list((RAW / "taxi_trips").glob("*.parquet"))
print("\ntaxi files:", [f.name for f in taxi_files])
taxi = spark.read.parquet(str(RAW / "taxi_trips"))
print("rows:", taxi.count())

taxi.printSchema()
taxi.show(3, vertical=True)


taxi files: ['yellow_tripdata_2024-01.parquet', 'yellow_tripdata_2024-02.parquet', 'yellow_tripdata_2024-03.parquet']
rows: 9554778
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable =

In [24]:
# Air Quality

aq = pd.read_csv(RAW / "air_quality" / "hourly_88101_2024.csv")

print("AQ rows:", len(aq))
print("AQ date range:", aq["Date Local"].min(), "→", aq["Date Local"].max())
print("AQ sites:", aq.groupby(["State Code", "County Code", "Site Num"]).ngroups)
print("AQ nulls in Sample Measurement:", aq["Sample Measurement"].isna().sum())
print("AQ nulls in Time Local:", aq["Time Local"].isna().sum())
print(
    "AQ dupes on site+datetime:",
    aq.duplicated(
        subset=[
            "State Code",
            "County Code",
            "Site Num",
            "POC",
            "Date Local",
            "Time Local",
        ]
    ).sum(),
)

C:\Users\user\AppData\Local\Temp\ipykernel_15012\3677814887.py:3: DtypeWarning: Columns (0: Qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  aq = pd.read_csv(RAW / "air_quality" / "hourly_88101_2024.csv")


AQ rows: 8139551
AQ date range: 2024-01-01 → 2024-12-31
AQ sites: 925
AQ nulls in Sample Measurement: 0
AQ nulls in Time Local: 0
AQ dupes on site+datetime: 0


In [25]:
# Weather

w = pd.read_csv(RAW / "weather" / "weather.csv")

print("Weather rows:", len(w))
print(
    "Weather date range:",
    f"{w['year'].min()}-{w['month'].min():02d}-{w['day'].min():02d} → "
    f"{w['year'].max()}-{w['month'].max():02d}-{w['day'].max():02d}",
)

print(
    "Weather hours per day:",
    w.groupby(["year", "month", "day"])["hour"].nunique().min(),
    "to",
    w.groupby(["year", "month", "day"])["hour"].nunique().max(),
)

print(
    "Weather dupes on year/month/day/hour:",
    w.duplicated(subset=["year", "month", "day", "hour"]).sum(),
)

Weather rows: 8784
Weather date range: 2024-01-01 → 2024-12-31
Weather hours per day: 24 to 24
Weather dupes on year/month/day/hour: 0


In [26]:
# Taxi Zones

tz = pd.read_csv(RAW / "taxi_zones" / "taxi_zone_lookup.csv")

print("Zones rows:", len(tz))
print("Zones dupes on LocationID:", tz["LocationID"].duplicated().sum())
print("Zones null LocationID:", tz["LocationID"].isna().sum())
print("Boroughs:", sorted(tz["Borough"].dropna().unique()))
print("Zones with null Borough:", tz["Borough"].isna().sum())
print("Zones with null Zone:", tz["Zone"].isna().sum())

Zones rows: 265
Zones dupes on LocationID: 0
Zones null LocationID: 0
Boroughs: ['Bronx', 'Brooklyn', 'EWR', 'Manhattan', 'Queens', 'Staten Island', 'Unknown']
Zones with null Borough: 1
Zones with null Zone: 1
